In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/03_model_training_dataset.csv")   # change to your file name / path

print(df.shape)
print(df.dtypes)
print(df.head(10).to_string())
print(df.describe().T.to_string())

print("\nMissing per column:")
print(df.isna().sum())

print("\nYear range:", df["year"].min(), "-", df["year"].max())
print("Unique locations:", df.groupby(["lat","lon"]).ngroups)
print("Rows per location (min/median/max):")
print(df.groupby(["lat","lon"]).size().describe())

(252031, 16)
year                                int64
lat                               float64
lon                               float64
month_sin                         float64
month_cos                         float64
rainfall_anomaly_lag_1            float64
rainfall_anomaly_lag_2            float64
rainfall_anomaly_lag_3            float64
rainfall_anomaly_rolling_3m       float64
rainfall_anomaly_rolling_6m       float64
rainfall_anomaly_std_3m           float64
temperature_anomaly_lag_1         float64
temperature_anomaly_lag_2         float64
temperature_anomaly_lag_3         float64
temperature_anomaly_rolling_3m    float64
rainfall_anomaly                  float64
dtype: object
   year  lat   lon     month_sin     month_cos  rainfall_anomaly_lag_1  rainfall_anomaly_lag_2  rainfall_anomaly_lag_3  rainfall_anomaly_rolling_3m  rainfall_anomaly_rolling_6m  rainfall_anomaly_std_3m  temperature_anomaly_lag_1  temperature_anomaly_lag_2  temperature_anomaly_lag_3  temperature_anoma

In [1]:
# add month and target 2 3 4 month columns
import numpy as np, pandas as pd
from pathlib import Path

PROC = Path("../data/processed")
print(sorted(p.name for p in PROC.iterdir()))

FILE = PROC/"03_model_training_dataset.csv"
df = pd.read_csv(FILE)

# recover month from sin/cos
ang = np.arctan2(df["month_sin"], df["month_cos"]) % (2*np.pi)
df["month"] = ((np.round(ang*12/(2*np.pi)).astype(int) - 1) % 12) + 1
df["t"] = df["year"]*12 + df["month"]            # integer month index
df["date"] = pd.to_datetime(dict(year=df.year, month=df.month, day=1))
df = df.sort_values(["lat","lon","t"]).reset_index(drop=True)

# --- sanity checks ---
print("date range:", df.date.min().date(), "->", df.date.max().date())
print("duplicate rows:", df.duplicated(["lat","lon","t"]).sum())
g = df.groupby(["lat","lon"])
consec = g["t"].diff() == 1
prev_target = g["rainfall_anomaly"].shift(1)
print("lag_1 matches previous month's anomaly:",
      np.allclose(df.loc[consec, "rainfall_anomaly_lag_1"], prev_target[consec]))
gaps = (g["t"].diff() > 1).groupby([df.lat, df.lon]).sum()
print("locations with gaps:", (gaps > 0).sum(), "| total gap points:", gaps.sum())

# --- lead targets (matched on exact month, so gaps are handled correctly) ---
df = df.rename(columns={"rainfall_anomaly": "target_t+1"})
for h in (2, 3, 4):
    s = df[["lat","lon","t","target_t+1"]].copy()
    s["t"] = s["t"] - (h - 1)
    s = s.rename(columns={"target_t+1": f"target_t+{h}"})
    df = df.merge(s, on=["lat","lon","t"], how="left")

print(df[["target_t+1","target_t+3","target_t+4"]].isna().sum())
print(df[["year","month","lat","lon","rainfall_anomaly_lag_1",
          "target_t+1","target_t+3","target_t+4"]].head(8).to_string())

df.to_csv(PROC / "model_dataset_with_targets.csv", index=False)

['01_consolidated_before_final_feature_alignment.csv', '01_initial_consolidation.csv', '02_before_calendar_alignment.csv', '02_feature_engineering_working.csv', '03_model_training_dataset.csv', 'location_lookup.csv', 'locations_with_missing_climate_values.csv', 'missing_values_consolidated.csv', 'trial_1951.csv']
date range: 1951-07-01 -> 2025-12-01
duplicate rows: 0
lag_1 matches previous month's anomaly: True
locations with gaps: 8 | total gap points: 27
target_t+1      0
target_t+3    620
target_t+4    929
dtype: int64
   year  month  lat   lon  rainfall_anomaly_lag_1  target_t+1  target_t+3  target_t+4
0  1951      7  8.5  77.5               43.484516  -22.638708   76.465412  -70.360432
1  1951      8  8.5  77.5              -22.638708  -34.401511  -70.360432  117.969890
2  1951      9  8.5  77.5              -34.401511   76.465412  117.969890  -67.152160
3  1951     10  8.5  77.5               76.465412  -70.360432  -67.152160   -7.073774
4  1951     11  8.5  77.5              -70

In [3]:
print(df.shape)
print(list(df.columns))
print(df.head(8).to_string())

tcols = [c for c in df.columns if c.startswith("target")]
print("\nNaN per target:")
print(df[tcols].isna().sum())

(252031, 22)
['year', 'lat', 'lon', 'month_sin', 'month_cos', 'rainfall_anomaly_lag_1', 'rainfall_anomaly_lag_2', 'rainfall_anomaly_lag_3', 'rainfall_anomaly_rolling_3m', 'rainfall_anomaly_rolling_6m', 'rainfall_anomaly_std_3m', 'temperature_anomaly_lag_1', 'temperature_anomaly_lag_2', 'temperature_anomaly_lag_3', 'temperature_anomaly_rolling_3m', 'target_t+1', 'month', 't', 'date', 'target_t+2', 'target_t+3', 'target_t+4']
   year  lat   lon     month_sin     month_cos  rainfall_anomaly_lag_1  rainfall_anomaly_lag_2  rainfall_anomaly_lag_3  rainfall_anomaly_rolling_3m  rainfall_anomaly_rolling_6m  rainfall_anomaly_std_3m  temperature_anomaly_lag_1  temperature_anomaly_lag_2  temperature_anomaly_lag_3  temperature_anomaly_rolling_3m  target_t+1  month      t       date  target_t+2  target_t+3  target_t+4
0  1951  8.5  77.5 -5.000000e-01 -8.660254e-01               43.484516               22.288481               89.001956                    51.591651                    23.505317        

In [5]:
import numpy as np, pandas as pd
from pathlib import Path

PROC = Path("../data/processed")
df = pd.read_csv(PROC / "model_dataset_with_targets.csv")

lines = []
def log(x=""):
    lines.append(str(x))

TRAIN_END = 2002*12 + 12
VAL_END   = 2013*12 + 12
FLOOR_MM  = 5.0

# climatological std per cell and calendar month, from training years only
sd = (df[df.t <= TRAIN_END]
        .groupby(["lat","lon","month"])["target_t+1"].std()
        .rename("sd").reset_index())
log("cell-month std (mm) quantiles:")
log(sd["sd"].quantile([0, .01, .05, .25, .5, .75, .95, 1]).round(2).to_string())
log(f"share of cell-months with std < 5 mm: {(sd['sd'] < FLOOR_MM).mean():.3f}")

# standardized targets: target month for lead h is M + h - 1
for h in (1, 2, 3, 4):
    tm = ((df["month"] + h - 2) % 12) + 1
    key = df[["lat","lon"]].assign(month=tm)
    sdv = key.merge(sd, on=["lat","lon","month"], how="left")["sd"].values
    df[f"sd_t+{h}"] = np.maximum(sdv, FLOOR_MM)
    df[f"z_t+{h}"]  = df[f"target_t+{h}"] / df[f"sd_t+{h}"]

# split flags with buffer
df["split"] = np.where(df.t <= TRAIN_END, "train",
              np.where(df.t <= VAL_END, "val", "test"))
for end in (TRAIN_END, VAL_END):
    df.loc[(df.t > end - 3) & (df.t <= end), "split"] = "buffer"

log("\nrows per split:")
log(df["split"].value_counts().to_string())
log("\nz_t+3 by split:")
log(df.groupby("split")["z_t+3"].describe().round(2).to_string())
log(f"\n|z_t+3| > 6: {(df['z_t+3'].abs() > 6).sum()} | max |z|: {df['z_t+3'].abs().max():.1f}")
log("\nseasonal z_t+3 std by target month:")
tm3 = ((df["month"] + 1) % 12) + 1
log(df.groupby(tm3)["z_t+3"].std().round(2).to_string())
log("\nNaN in z columns: " + str(df[[f"z_t+{h}" for h in (1,2,3,4)]].isna().sum().to_dict()))
log("new columns: " + str([c for c in df.columns if c.startswith(("sd_","z_"))] + ["split"]))

Path("step3_report.txt").write_text("\n".join(lines))
df.to_csv(PROC / "model_dataset_split.csv", index=False)
print("\n".join(lines))

cell-month std (mm) quantiles:
0.00      0.45
0.01      1.66
0.05      4.64
0.25     14.01
0.50     34.70
0.75     79.63
0.95    138.04
1.00    556.66
share of cell-months with std < 5 mm: 0.057

rows per split:
split
train     173106
test       40752
val        36481
buffer      1692

z_t+3 by split:
           count  mean   std   min   25%   50%   75%    max
split                                                      
buffer    1692.0  0.09  1.13 -1.32 -0.56 -0.32  0.28   9.08
test     40186.0  0.10  1.23 -3.06 -0.60 -0.24  0.50  27.21
train   173056.0 -0.01  0.98 -3.04 -0.61 -0.27  0.36   7.14
val      36477.0  0.02  1.21 -3.62 -0.63 -0.30  0.34  26.08

|z_t+3| > 6: 439 | max |z|: 27.2

seasonal z_t+3 std by target month:
month
1     0.98
2     0.98
3     1.28
4     1.05
5     1.15
6     1.10
7     1.04
8     1.06
9     1.02
10    0.98
11    1.00
12    1.05

NaN in z columns: {'z_t+1': 0, 'z_t+2': 310, 'z_t+3': 620, 'z_t+4': 929}
new columns: ['sd_t+1', 'z_t+1', 'sd_t+2', 'z_t+2', 's

In [6]:
df = pd.read_csv(PROC / "model_dataset_split.csv")
big = df[df["z_t+3"].abs() > 6].copy()
big["target_month"] = ((big["month"] + 1) % 12) + 1

print(big["split"].value_counts().to_string())
print("\nby target month:\n", big.groupby("target_month").size().to_string())
print("\nsd at floor (5 mm):", (big["sd_t+3"] == 5.0).mean().round(2))
print("\ntop cells:\n", big.groupby(["lat","lon"]).size().sort_values(ascending=False).head(8).to_string())
print("\nlargest:")
print(big.assign(mm=big["target_t+3"], sd=big["sd_t+3"], z=big["z_t+3"])
         [["lat","lon","date","target_month","mm","sd","z"]]
         .sort_values("z", key=abs, ascending=False).head(8).to_string())

split
test      190
val       153
train      89
buffer      7

by target month:
 target_month
1      28
2      15
3     138
4      34
5      75
6      27
7       7
8       9
9       7
10     15
11     32
12     52

sd at floor (5 mm): 0.14

top cells:
 lat   lon 
18.5  74.5    8
      82.5    7
21.5  71.5    7
20.5  76.5    6
9.5   78.5    6
25.5  73.5    6
12.5  75.5    5
13.5  75.5    5

largest:
        lat   lon        date  target_month          mm        sd          z
82190  21.5  71.5  2021-03-01             5  136.060407  5.000000  27.212081
26604  15.5  75.5  2008-01-01             3  139.349914  5.342704  26.082284
47355  17.5  82.5  2023-10-01            12  210.419738  8.347792  25.206633
48954  18.5  74.5  2008-01-01             3  117.682433  5.000000  23.536487
27498  15.5  76.5  2008-01-01             3  111.242226  5.000000  22.248445
20346  14.5  74.5  2008-01-01             3  103.281062  5.000000  20.656212
81344  21.5  70.5  2025-03-01             5  101.533767  5.

New Columns added : 

- month: calendar month (1-12) of the row, recovered from month_sin/cos.
- t: integer month index (year*12 + month), used for shifting and splitting.
- date: first day of that month, for bookkeeping only (not a feature).
- target_t+1 to target_t+4: rainfall anomaly (mm) 1, 2, 3 and 4 months after the forecast origin.
- sd_t+1 to sd_t+4: training-years std (mm) of the target month for that cell, floored at 5 mm.
- z_t+1 to z_t+4: standardized target, target_t+h / sd_t+h (the target we’ll train on).
- split: train (to Dec 2002), val (2003-2013), test (2014-2025), or buffer (3 rows before each boundary, excluded from training and evaluation).

In [7]:
new_cols = ["year", "month", "t", "date", "lat", "lon",
            "target_t+1", "target_t+2", "target_t+3", "target_t+4"]

print(df.shape)
print(df[new_cols].head(10).to_string())
print("\nLast rows of the first cell:")
print(df[new_cols].iloc[890:896].to_string())

(252031, 31)
   year  month      t        date  lat   lon  target_t+1  target_t+2  target_t+3  target_t+4
0  1951      7  23419  1951-07-01  8.5  77.5  -22.638708  -34.401511   76.465412  -70.360432
1  1951      8  23420  1951-08-01  8.5  77.5  -34.401511   76.465412  -70.360432  117.969890
2  1951      9  23421  1951-09-01  8.5  77.5   76.465412  -70.360432  117.969890  -67.152160
3  1951     10  23422  1951-10-01  8.5  77.5  -70.360432  117.969890  -67.152160   -7.073774
4  1951     11  23423  1951-11-01  8.5  77.5  117.969890  -67.152160   -7.073774   24.353454
5  1951     12  23424  1951-12-01  8.5  77.5  -67.152160   -7.073774   24.353454  -23.261432
6  1952      1  23425  1952-01-01  8.5  77.5   -7.073774   24.353454  -23.261432   17.869072
7  1952      2  23426  1952-02-01  8.5  77.5   24.353454  -23.261432   17.869072    9.899998
8  1952      3  23427  1952-03-01  8.5  77.5  -23.261432   17.869072    9.899998  -36.759325
9  1952      4  23428  1952-04-01  8.5  77.5   17.869072 

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

PROC = Path("../data/processed")
RAW  = Path("../data/raw")                      # edit to your raw folder
DATASET = PROC / "model_dataset_split.csv"
NINO    = RAW / "nino34.long.anom.csv"                    # edit to your Nino 3.4 file name
DMI     = RAW / "dmi.had.long.csv"                       # edit to your DMI file name
OUT     = PROC / "model_dataset_with_enso.csv"  # new file, the input is not overwritten

MISSING_CUTOFF = -99.0   # catches -99.99 (Nino) and -9999 (DMI)

def read_psl_series(path, name):
    """Read a NOAA PSL monthly series. Skips the header line, footers and blank lines."""
    rows = []
    for line in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 2:
            continue
        d = pd.to_datetime(parts[0], format="%Y-%m-%d", errors="coerce")
        if pd.isna(d):
            continue
        try:
            v = float(parts[1].split()[0])
        except (ValueError, IndexError):
            continue
        rows.append((d.year * 12 + d.month, v))
    s = pd.DataFrame(rows, columns=["t", name])
    s.loc[s[name] <= MISSING_CUTOFF, name] = np.nan
    assert not s["t"].duplicated().any(), f"{path}: duplicate months"
    return s

def add_lags(df, series, name):
    """lag_k = value at month t-k, looked up by exact month number. mean_3m needs all 3 lags."""
    lookup = series.set_index("t")[name]
    lag_cols = []
    for k in (1, 2, 3):
        col = f"{name}_lag_{k}"
        df[col] = (df["t"] - k).map(lookup)
        lag_cols.append(col)
    df[f"{name}_mean_3m"] = df[lag_cols].mean(axis=1, skipna=False)
    return df

df = pd.read_csv(DATASET)
n_rows, n_cols = df.shape
print("Loaded", DATASET.name, df.shape)
print("Has split column:", "split" in df.columns, "| has t column:", "t" in df.columns)

if "t" not in df.columns:
    df["t"] = df["year"] * 12 + df["month"]
assert (df["t"] == df["year"] * 12 + df["month"]).all(), "t != year*12+month"

nino_s = read_psl_series(NINO, "nino34")
dmi_s  = read_psl_series(DMI, "dmi")

for s, nm in ((nino_s, "nino34"), (dmi_s, "dmi")):
    first, last = s["t"].min(), s["t"].max()
    fmt = lambda t: f"{(t - 1) // 12}-{(t - 1) % 12 + 1:02d}"
    print(f"{nm}: {len(s)} months, {fmt(first)} to {fmt(last)}, missing values: {s[nm].isna().sum()}")
    n_short = int((df["t"] - 1 > last).sum())
    if n_short:
        print(f"  WARNING: {nm} ends too early, {n_short} rows will have NaN lag_1")

df = add_lags(df, nino_s, "nino34")
df = add_lags(df, dmi_s, "dmi")

new_cols = [c for c in df.columns if c.startswith(("nino34_", "dmi_"))]
assert len(df) == n_rows and len(new_cols) == 8

print("\nNaN count in new columns:")
print(df[new_cols].isna().sum().to_string())

if "split" in df.columns:
    print("\nRows with any NaN in the new columns, by split:")
    print(df.assign(_na=df[new_cols].isna().any(axis=1)).groupby("split")["_na"].sum().to_string())

# Sanity checks, compare against the source files by eye
one = df.drop_duplicates("t").set_index("t")
for label, y, m in (("Dec 1997 (origin Nov 1997)", 1997, 12), ("Nov 2010 (origin Oct 2010)", 2010, 11)):
    t = y * 12 + m
    if t in one.index:
        print(f"{label}: nino34_lag_1={one.loc[t, 'nino34_lag_1']:.2f}, dmi_lag_1={one.loc[t, 'dmi_lag_1']:.2f}")

# Each month must carry one identical value across all cells
assert (df.groupby("t")["nino34_lag_1"].nunique(dropna=True) <= 1).all()
assert (df.groupby("t")["dmi_lag_1"].nunique(dropna=True) <= 1).all()

df.to_csv(OUT, index=False)
print(f"\nSaved {OUT.name}: {df.shape[0]} rows x {df.shape[1]} cols (was {n_cols})")

Loaded model_dataset_split.csv (252031, 31)
Has split column: True | has t column: True
nino34: 1884 months, 1870-01 to 2026-12, missing values: 3
dmi: 1884 months, 1870-01 to 2026-12, missing values: 5

NaN count in new columns:
nino34_lag_1      0
nino34_lag_2      0
nino34_lag_3      0
nino34_mean_3m    0
dmi_lag_1         0
dmi_lag_2         0
dmi_lag_3         0
dmi_mean_3m       0

Rows with any NaN in the new columns, by split:
split
buffer    0
test      0
train     0
val       0
Dec 1997 (origin Nov 1997): nino34_lag_1=2.42, dmi_lag_1=1.28
Nov 2010 (origin Oct 2010): nino34_lag_1=-1.65, dmi_lag_1=-0.44

Saved model_dataset_with_enso.csv: 252031 rows x 39 cols (was 31)


In [12]:
df = pd.read_csv(PROC / "model_dataset_with_enso.csv")
print(df.shape)
print(list(df.columns))
print(df.dtypes.to_string())
df.head(5).T

(252031, 39)
['year', 'lat', 'lon', 'month_sin', 'month_cos', 'rainfall_anomaly_lag_1', 'rainfall_anomaly_lag_2', 'rainfall_anomaly_lag_3', 'rainfall_anomaly_rolling_3m', 'rainfall_anomaly_rolling_6m', 'rainfall_anomaly_std_3m', 'temperature_anomaly_lag_1', 'temperature_anomaly_lag_2', 'temperature_anomaly_lag_3', 'temperature_anomaly_rolling_3m', 'target_t+1', 'month', 't', 'date', 'target_t+2', 'target_t+3', 'target_t+4', 'sd_t+1', 'z_t+1', 'sd_t+2', 'z_t+2', 'sd_t+3', 'z_t+3', 'sd_t+4', 'z_t+4', 'split', 'nino34_lag_1', 'nino34_lag_2', 'nino34_lag_3', 'nino34_mean_3m', 'dmi_lag_1', 'dmi_lag_2', 'dmi_lag_3', 'dmi_mean_3m']
year                                int64
lat                               float64
lon                               float64
month_sin                         float64
month_cos                         float64
rainfall_anomaly_lag_1            float64
rainfall_anomaly_lag_2            float64
rainfall_anomaly_lag_3            float64
rainfall_anomaly_rolling_3m    

,0,1,2,3,4
year,1951,1951,1951,1951,1951
lat,8.5,8.5,8.5,8.5,8.5
lon,77.5,77.5,77.5,77.5,77.5
month_sin,-0.5,-0.866025,-1.0,-0.866025,-0.5
month_cos,-0.866025,-0.5,-0.0,0.5,0.866025
rainfall_anomaly_lag_1,43.484516,-22.638708,-34.401511,76.465412,-70.360432
rainfall_anomaly_lag_2,22.288481,43.484516,-22.638708,-34.401511,76.465412
rainfall_anomaly_lag_3,89.001956,22.288481,43.484516,-22.638708,-34.401511
rainfall_anomaly_rolling_3m,51.591651,14.378096,-4.518568,6.475064,-9.432177
rainfall_anomaly_rolling_6m,23.505317,16.766826,14.470587,29.033358,2.47296


In [14]:
import json

df = pd.read_csv(PROC / "model_dataset_with_enso.csv")

FEATURES = [
    "lat", "lon",
    "month_sin", "month_cos",
    "rainfall_anomaly_lag_1", "rainfall_anomaly_lag_2", "rainfall_anomaly_lag_3",
    "rainfall_anomaly_rolling_3m", "rainfall_anomaly_rolling_6m", "rainfall_anomaly_std_3m",
    "temperature_anomaly_lag_1", "temperature_anomaly_lag_2", "temperature_anomaly_lag_3",
    "temperature_anomaly_rolling_3m",
    "nino34_lag_1", "nino34_lag_2", "nino34_lag_3", "nino34_mean_3m",
    "dmi_lag_1", "dmi_lag_2", "dmi_lag_3", "dmi_mean_3m",
]
assert all(c in df.columns for c in FEATURES)
assert not any(c.startswith(("target_", "sd_", "z_")) or c in ("year", "t", "date", "split") for c in FEATURES)

print("Before:", df["split"].value_counts().to_dict())

df = df[df["split"] != "buffer"].copy()
print("After dropping buffer:", df["split"].value_counts().to_dict())

na_by_col = df[FEATURES].isna().sum()
print("\nNaN per input column (non-buffer rows):")
print(na_by_col[na_by_col > 0].to_string() if (na_by_col > 0).any() else "none")

n0 = len(df)
df = df.dropna(subset=FEATURES).copy()
print(f"\nDropped {n0 - len(df)} rows with NaN inputs")
print("Final:", df["split"].value_counts().to_dict(), "| total", len(df))

for h in (1, 2, 3, 4):
    print(f"z_t+{h} NaN by split:", df.groupby("split")[f"z_t+{h}"].apply(lambda s: s.isna().sum()).to_dict())

df.to_csv(PROC / "model_dataset_final.csv", index=False)
json.dump({"features": FEATURES, "targets": [f"z_t+{h}" for h in (1, 2, 3, 4)]},
          open(PROC / "feature_list.json", "w"), indent=2)

Before: {'train': 173106, 'test': 40752, 'val': 36481, 'buffer': 1692}
After dropping buffer: {'train': 173106, 'test': 40752, 'val': 36481}

NaN per input column (non-buffer rows):
none

Dropped 0 rows with NaN inputs
Final: {'train': 173106, 'test': 40752, 'val': 36481} | total 250339
z_t+1 NaN by split: {'test': 0, 'train': 0, 'val': 0}
z_t+2 NaN by split: {'test': 283, 'train': 25, 'val': 2}
z_t+3 NaN by split: {'test': 566, 'train': 50, 'val': 4}
z_t+4 NaN by split: {'test': 849, 'train': 74, 'val': 6}
